# Introduction to Vector Databases (Installing and Running ChromaDB)**Level:** Scratch → Intermediate**Runs on:** Google Colab (Free CPU)### What you will learn1. What a Vector Database is and why we need one2. Installing ChromaDB (free, open-source, runs locally)3. Creating a Collection (like a table, but for vectors)4. Adding documents (ChromaDB auto-generates embeddings)5. Querying by meaning (semantic search)6. Using your own Hugging Face embeddings with ChromaDB (intermediate)7. Updating and deleting records> No API key, no cloud account, no cost. Everything runs in-memory inside this notebook.

## 1. Why a Vector Database?A normal database (SQL) searches by **exact match** — e.g., `WHERE name = 'dog'`.A **Vector Database** stores embeddings (numeric meaning-vectors) and searches by **similarity** — e.g., "find text that *means* something like 'dog'", even if the word "dog" never appears.| Normal Database | Vector Database ||---|---|| Matches exact text/values | Matches by meaning || Uses SQL queries | Uses similarity search || Good for structured data | Good for text, images, audio embeddings |**ChromaDB** is one of the simplest free, open-source vector databases — great for learning.

### 2. Install ChromaDBRun this once. `-q` keeps the output clean.

In [ ]:
!pip install -q chromadb

### 3. Create a ChromaDB ClientWe use an **in-memory / ephemeral client** — perfect for learning in Colab.(Data will reset when the runtime restarts. For permanent storage, `chromadb.PersistentClient(path="...")` is used instead — shown later.)

In [ ]:
import chromadbclient = chromadb.Client()print("ChromaDB client created successfully!")

### 4. Create a CollectionA **Collection** in ChromaDB is like a table — it stores your documents, their embeddings, and metadata.

In [ ]:
collection = client.create_collection(name="my_first_collection")print("Collection created:", collection.name)

### 5. Add DocumentsWe simply give ChromaDB plain text. By default, it **automatically converts text into embeddings** for us using a built-in free model — no extra setup needed.

In [ ]:
collection.add(    documents=[        "Python is a popular programming language for AI.",        "The weather today is sunny and warm.",        "Machine learning models learn patterns from data.",        "I enjoy playing football on weekends.",        "Deep learning uses neural networks with many layers.",        "Paris is the capital city of France."    ],    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6"])print("Documents added! Total documents:", collection.count())

**Discussion Point:** Every document got an `id` (like a primary key). Behind the scenes, ChromaDB converted each sentence into an embedding vector and stored it — just like we did manually in the earlier embeddings notebook.

### 6. Query by Meaning (Semantic Search)Now we ask a question in natural language. ChromaDB converts our query into an embedding too, then finds the closest matching documents.

In [ ]:
results = collection.query(    query_texts=["Tell me about AI and neural networks"],    n_results=2)for doc, distance in zip(results["documents"][0], results["distances"][0]):    print(f"Distance: {distance:.4f}  |  {doc}")

**Discussion Point:** Lower `distance` = more similar (closer meaning). Notice the AI/ML related sentences come back first, even though the query never used the exact same words.

In [ ]:
# Try another queryresults = collection.query(    query_texts=["What is the capital of France?"],    n_results=2)for doc, distance in zip(results["documents"][0], results["distances"][0]):    print(f"Distance: {distance:.4f}  |  {doc}")

### 7. Adding Metadata (Intermediate)Real-world documents usually have extra info (category, author, date). ChromaDB lets us store this alongside each document and filter on it.

In [ ]:
collection.add(    documents=[        "The new smartphone has an amazing camera.",        "Stock prices rose sharply after the earnings report."    ],    metadatas=[        {"category": "technology"},        {"category": "finance"}    ],    ids=["doc7", "doc8"])# Query only within the "technology" categoryresults = collection.query(    query_texts=["Tell me about new gadgets"],    n_results=2,    where={"category": "technology"})for doc, distance in zip(results["documents"][0], results["distances"][0]):    print(f"Distance: {distance:.4f}  |  {doc}")

### 8. Using Your Own Hugging Face Embeddings (Intermediate)By default ChromaDB uses its own built-in embedding model. But we can plug in **any** Hugging Face model, such as `all-MiniLM-L6-v2` from the earlier notebook, for full control.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from chromadb.utils import embedding_functionshf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(    model_name="all-MiniLM-L6-v2")# Create a new collection that uses our chosen Hugging Face modelcustom_collection = client.create_collection(    name="custom_embedding_collection",    embedding_function=hf_embedder)custom_collection.add(    documents=[        "I love dogs",        "I like puppies",        "The stock market crashed today"    ],    ids=["a1", "a2", "a3"])results = custom_collection.query(    query_texts=["puppy lovers"],    n_results=2)for doc, distance in zip(results["documents"][0], results["distances"][0]):    print(f"Distance: {distance:.4f}  |  {doc}")

### 9. Update and Delete Records

In [ ]:
# Update a document's textcollection.update(    ids=["doc4"],    documents=["I enjoy playing football and basketball on weekends."])print("Updated doc4")# Delete a documentcollection.delete(ids=["doc2"])print("Deleted doc2")print("Remaining documents:", collection.count())

### 10. Persistent Storage (So Data Survives a Restart)The examples above used an in-memory client, which resets on restart. For real projects, use `PersistentClient` to save data to disk.

In [ ]:
persistent_client = chromadb.PersistentClient(path="/content/chroma_db")persistent_collection = persistent_client.get_or_create_collection(name="persistent_demo")persistent_collection.add(    documents=["This data will be saved to disk."],    ids=["p1"])print("Saved to disk at /content/chroma_db")print("Count:", persistent_collection.count())

### 11. Key Takeaways (Recap for Students)| Concept | Meaning ||---|---|| Vector Database | Stores embeddings, searches by meaning/similarity || ChromaDB | Free, open-source, runs locally — no cloud needed || Collection | Like a table — holds documents + embeddings + metadata || `add()` | Insert documents (embeddings created automatically) || `query()` | Search by meaning, returns closest matches || `where={}` | Filter results using metadata || `PersistentClient` | Saves the database to disk instead of memory |### 12. Practice Exercises1. Add 5 of your own sentences to a new collection and query them with a related question.2. Add a `"category"` metadata field to every document, then filter a query by two different categories.3. Swap the embedding function to `'all-mpnet-base-v2'` and compare the distance scores to `all-MiniLM-L6-v2`.

## 12. Practice Exercise Answers

In [ ]:
# Exercise 1: Add 5 custom documents and perform a semantic search
exercise_collection = client.get_or_create_collection(name="exercise_collection")
exercise_collection.add(
    documents=[
        "Python is useful for data analysis.",
        "Deep learning uses neural networks.",
        "Football is a popular sport.",
        "The weather is sunny today.",
        "Paris is the capital of France."
    ],
    ids=["e1", "e2", "e3", "e4", "e5"]
)

results = exercise_collection.query(
    query_texts=["Tell me about neural networks and AI"],
    n_results=2
)
for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | {doc}")

In [ ]:
# Exercise 2: Store categories and filter using two categories
category_collection = client.get_or_create_collection(name="category_exercise")
category_collection.add(
    documents=[
        "Python programming tutorial",
        "Deep learning neural networks",
        "Football match this weekend",
        "Basketball training tips",
        "Machine learning with Python"
    ],
    metadatas=[
        {"category": "technology"},
        {"category": "technology"},
        {"category": "sports"},
        {"category": "sports"},
        {"category": "technology"}
    ],
    ids=["c1", "c2", "c3", "c4", "c5"]
)

results = category_collection.query(
    query_texts=["learning and programming"],
    n_results=5,
    where={"category": {"$in": ["technology", "sports"]}}
)
for doc, meta, distance in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"{meta['category']} | {distance:.4f} | {doc}")

In [ ]:
# Exercise 3: Compare all-MiniLM-L6-v2 with all-mpnet-base-v2
from chromadb.utils import embedding_functions

mini_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
mpnet_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

mini_collection = client.get_or_create_collection(
    name="mini_compare",
    embedding_function=mini_embedder
)
mpnet_collection = client.get_or_create_collection(
    name="mpnet_compare",
    embedding_function=mpnet_embedder
)

compare_docs = ["I love dogs", "I like puppies", "The stock market crashed today"]
compare_ids = ["m1", "m2", "m3"]
mini_collection.add(documents=compare_docs, ids=compare_ids)
mpnet_collection.add(documents=compare_docs, ids=compare_ids)

query = "puppy lovers"
mini_result = mini_collection.query(query_texts=[query], n_results=3)
mpnet_result = mpnet_collection.query(query_texts=[query], n_results=3)

print("all-MiniLM-L6-v2 distances:")
for doc, dist in zip(mini_result["documents"][0], mini_result["distances"][0]):
    print(f"{dist:.4f} | {doc}")

print("\nall-mpnet-base-v2 distances:")
for doc, dist in zip(mpnet_result["documents"][0], mpnet_result["distances"][0]):
    print(f"{dist:.4f} | {doc}")